In [1]:
import torch

t = torch.tensor([1.0, 2.0, 3.0])
print(t)
print(t * 2)
print(t.sum())

tensor([1., 2., 3.])
tensor([2., 4., 6.])
tensor(6.)


In [5]:
import torch

a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

out = a * b
out.backward()

print("gradient of a:", a.grad)
print("gradient of b:", b.grad)

gradient of a: tensor(4.)
gradient of b: tensor(3.)


In [6]:
import torch
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

out = a + b
out.backward()

print("gradient of a:", a.grad)
print("gradient of b:", b.grad)

gradient of a: tensor(1.)
gradient of b: tensor(1.)


In [7]:
x1 = torch.tensor(2.0)
x2 = torch.tensor(-1.0)
w1 = torch.tensor(0.5, requires_grad=True)
w2 = torch.tensor(0.5, requires_grad=True)
b  = torch.tensor(0.0, requires_grad=True)

out = x1 * w1 + x2 * w2 + b
out.backward()

print("w1:", w1.grad)
print("w2:", w2.grad)
print("b:", b.grad)

w1: tensor(2.)
w2: tensor(-1.)
b: tensor(1.)


In [24]:
import torch

x = torch.tensor([1.0, 2.0, 3.0, 4.0])
y = torch.tensor([3.0, 6.0, 9.0, 12.0])

w = torch.tensor(0.0, requires_grad=True)   # the model starts knowing nothing
learning_rate = 0.05

for step in range(3):
    # 1. Forward: make predictions
    pred = w * x

    # 2. Loss: how wrong are we?
    loss = ((pred - y) ** 2).mean()

    # 3. Backward: work out the gradient
    loss.backward()

    # 4. Update: nudge w the opposite way to the gradient
    with torch.no_grad():
        w -= learning_rate * w.grad
    w.grad.zero_()

    print(f"step {step:2d}  w = {w.item():.4f}  loss = {loss.item():.4f}")

step  0  w = 2.2500  loss = 67.5000
step  1  w = 2.8125  loss = 4.2188
step  2  w = 2.9531  loss = 0.2637


In [9]:
import torch
import torch.nn as nn

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [6.0], [9.0], [12.0]])

model = nn.Linear(1, 1)                                   # replaces: w
loss_fn = nn.MSELoss()                                    # replaces: ((pred - y) ** 2).mean()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)  # replaces: the update line

for step in range(50):
    pred = model(x)              # 1. forward
    loss = loss_fn(pred, y)      # 2. loss
    optimizer.zero_grad()        #    reset gradients (was: w.grad.zero_())
    loss.backward()              # 3. backward
    optimizer.step()             # 4. update (was: w -= lr * w.grad)

    if step % 10 == 0:
        print(f"step {step:2d}  loss = {loss.item():.4f}")

print("learned weight:", model.weight.item())
print("learned bias:  ", model.bias.item())

step  0  loss = 66.0056
step 10  loss = 0.0568
step 20  loss = 0.0420
step 30  loss = 0.0311
step 40  loss = 0.0230
learned weight: 2.8914613723754883
learned bias:   0.31911712884902954


In [14]:
import torch
import torch.nn as nn

x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [6.0], [9.0], [12.0]])

model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for step in range(500):
    pred = model(x)              # 1. forward
    loss = loss_fn(pred, y)      # 2. loss
    optimizer.zero_grad()        #    reset gradients
    loss.backward()              # 3. backward
    optimizer.step()             # 4. update
    if step % 100 == 0:
        print(f"step {step:2d}  loss = {loss.item():.4f}")

print("learned weight:", model.weight.item())

print("learned bias:  ", model.bias.item())

step  0  loss = 80.1910
step 100  loss = 0.0004
step 200  loss = 0.0000
step 300  loss = 0.0000
step 400  loss = 0.0000
learned weight: 2.9999616146087646
learned bias:   0.00011310772970318794


In [20]:
import pandas as pd

df = pd.read_csv("fake_stock.csv")
gold = df[(df["metal"] == "Gold") & (df["purity"] == "22K")].copy()
print(len(gold), "items")
gold.head()


980 items


,metal,purity,weight_gm,branch,item_code,category
0,Gold,22K,16.20,Madurai,BA-2679,Bangle
1,Gold,22K,26.47,Chennai,EA-2535,Earring
2,Gold,22K,13.53,Trichy,EA-4611,Earring
5,Gold,22K,51.15,Coimbatore,BA-1711,Bangle
6,Gold,22K,6.57,Coimbatore,NE-6925,Necklace


In [21]:
import numpy as np

np.random.seed(42)
noise = np.random.uniform(-1, 1, len(gold))
gold["price_k"] = gold["weight_gm"] * 6.5 + 2 + noise

gold[["weight_gm", "price_k"]].head()

,weight_gm,price_k
0,16.20,107.049080
1,26.47,174.956429
2,13.53,90.408988
5,51.15,334.672317
6,6.57,44.017037


In [29]:
import torch

x = torch.tensor(gold[["weight_gm"]].values, dtype=torch.float32)
y = torch.tensor(gold[["price_k"]].values, dtype=torch.float32)
x_max = x.max()            # the biggest weight in the data
x_scaled = x / x_max
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for step in range(2000):
    pred = model(x_scaled)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 400 == 0:
        print(f"step {step:4d}  loss = {loss.item():.4f}")

learned_w = model.weight.item()
learned_b = model.bias.item()
print("learned weight:", learned_w)
print("learned bias:  ", learned_b)
print("rate per gram: ", learned_w / 60)

step    0  loss = 53959.9180
step  400  loss = 0.6832
step  800  loss = 0.3414
step 1200  loss = 0.3414
step 1600  loss = 0.3414
learned weight: 389.6314392089844
learned bias:   2.0028417110443115
rate per gram:  6.49385732014974


In [ ]:
import numpy as np

np.random.seed(42)
noise = np.random.uniform(-1, 1, len(gold))
gold["price_k"] = gold["weight_gm"] * 6.5 + 2 + noise

gold[["weight_gm", "price_k"]].head()

,weight_gm,price_k
0,16.20,107.049080
1,26.47,174.956429
2,13.53,90.408988
5,51.15,334.672317
6,6.57,44.017037


In [31]:
train = gold.sample(frac=0.8, random_state=42)
test = gold.drop(train.index)

print(len(train), "training items")
print(len(test), "test items")

784 training items
196 test items


In [32]:
x_train = torch.tensor(train[["weight_gm"]].values, dtype=torch.float32)
y_train = torch.tensor(train[["price_k"]].values, dtype=torch.float32)
x_test  = torch.tensor(test[["weight_gm"]].values, dtype=torch.float32)
y_test  = torch.tensor(test[["price_k"]].values, dtype=torch.float32)

x_max = x_train.max()           # from TRAINING data only
x_train_s = x_train / x_max
x_test_s  = x_test / x_max      # test scaled with the SAME number

In [33]:
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for step in range(2000):
    pred = model(x_train_s)
    loss = loss_fn(pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("training loss:", loss.item())

training loss: 0.33775243163108826


In [34]:
with torch.no_grad():
    test_pred = model(x_test_s)
    test_loss = loss_fn(test_pred, y_test)

print("test loss:    ", test_loss.item())

test loss:     0.35600849986076355


In [35]:
compare = test[["weight_gm", "price_k"]].head(5).copy()
compare["predicted"] = test_pred[:5].numpy().round(2)
compare

,weight_gm,price_k,predicted
1,26.47,174.956429,174.039993
6,6.57,44.017037,44.700001
57,52.59,343.259678,343.799988
63,54.17,353.468650,354.070007
105,14.85,98.748706,98.519997
